# Predictive Churn Analysis
Logistic Regression pipeline for customer churn prediction and risk scoring.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
df=pd.read_csv('churn_dataset.csv')
df.head()

## Preprocessing and 80/20 Split

In [ ]:
X=df.drop(columns=['CustomerID','Churn']); y=df['Churn']
num=['Age','TenureMonths','MonthlyCharges']
cat=['Contract','PaymentMethod','InternetService','TechSupport']
pre=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('scale',MinMaxScaler())]),num),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)])
model=Pipeline([('preprocess',pre),('model',LogisticRegression(max_iter=1000,class_weight='balanced',random_state=42))])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)
model.fit(X_train,y_train)

## Evaluation

In [ ]:
pred=model.predict(X_test)
proba=model.predict_proba(X_test)[:,1]
print(classification_report(y_test,pred))
print('ROC-AUC:',roc_auc_score(y_test,proba))

In [ ]:
fpr,tpr,_=roc_curve(y_test,proba)
plt.plot(fpr,tpr,label=f'AUC={roc_auc_score(y_test,proba):.3f}')
plt.plot([0,1],[0,1],'--'); plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('ROC-AUC Curve'); plt.legend(); plt.show()

## Churn Risk Scores

In [ ]:
all_proba=model.predict_proba(X)[:,1]
risk=np.where(all_proba>=0.70,'High',np.where(all_proba>=0.40,'Medium','Low'))
risk_scores=df[['CustomerID']].copy(); risk_scores['ChurnProbability']=all_proba.round(4); risk_scores['RiskLevel']=risk
risk_scores.to_csv('churn_risk_predictions.csv',index=False)
risk_scores.head(10)

## Conclusion
Use high-risk customers for targeted retention actions. The model is a decision-support tool and should be combined with business context.